# L03 · 量价关系 + 聚合

**预计时长**：60 min | **难度**：⭐⭐ | **前置**：L02

## 本节目标
1. 理解成交量、量价八法则、换手率
2. matplotlib 双轴图（收盘价 + 成交量）
3. `resample()` 按周/月聚合；`rolling()` 滚动均线
4. 处理停牌日 NaN（fillna / interpolate）

## 第 1 段：金融概念

 ### 成交量
 - 单位：**股数**（不是金额）。比亚迪 1 亿成交量 = 当天买卖了 1 亿股
 - 成交额 = volume × 平均价，akshare 默认不返回，自己算
 - 高成交量代表"分歧大"（有人大量买也有人大量卖）

 ### 量价八法则（核心 4 条）
 | 量价关系 | 含义 |
---------|------|
 | 量增价涨 | 健康上升，可持续 |
 | 量缩价涨 | 上升乏力，警惕 |
 | 量增价跌 | 抛压重，可能继续跌 |
 | 量缩价跌 | 抛压减弱，可能见底 |

 ### 换手率
 换手率 = 当日成交量 / 流通股本 × 100%
 - < 1%：低换手，盘面冷清
 - 1-5%：正常
 - 5-10%：活跃
 - > 10%：高度活跃（往往是热点题材或大单进出）
 - > 30%：极端（次新股、跌停封板出货常见）

 akshare 不直接给流通股本，要单独拉 `stock_individual_info_em`，本节略。

In [ ]:
import sys
from pathlib import Path

# 自动定位 phase1_foundation 目录 + project root（兼容两种 jupyter 启动位置）
_cwd = Path.cwd()
_p1 = _cwd if (_cwd / '_data.py').exists() else (_cwd / 'learning' / 'phase1_foundation')
sys.path.insert(0, str(_p1))
_proj = _p1.parent.parent if _p1.name == 'phase1_foundation' else _p1
if (_proj / 'qtrader' / '__init__.py').exists():
    sys.path.insert(0, str(_proj))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import _style
from _data import get_stock_data

In [ ]:
byd = get_stock_data('002594')

## 第 2 段：双轴图（价格 + 成交量）

In [ ]:
df = byd.set_index('date').tail(120)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})
ax1.plot(df.index, df['close'], color=_style.COLORS['price'], label='收盘价')
ax1.set_ylabel('价格（元）')
ax1.legend(loc='upper left')
ax1.grid(alpha=0.3)

ax2.bar(df.index, df['volume'], color=_style.COLORS['volume'], width=0.8)
ax2.set_ylabel('成交量（股）')
ax2.grid(alpha=0.3)

plt.suptitle('比亚迪 价量双轴图（近 120 日）')
plt.tight_layout()
plt.show()

## 第 3 段：`resample()` 按周/月聚合

In [ ]:
# 必须 set_index('date') 才能 resample
byd_idx = byd.set_index('date')

# 周线：W = weekly；OHLC 用 ohlc() 聚合，volume 用 sum()
weekly = byd_idx.resample('W').agg({
    'open': 'first', 'high': 'max', 'low': 'min',
    'close': 'last', 'volume': 'sum'
})
print(f"日线 {len(byd_idx)} 行 → 周线 {len(weekly)} 行")
weekly.tail(5)

In [ ]:
# 月线
monthly = byd_idx.resample('M').agg({
    'open': 'first', 'high': 'max', 'low': 'min',
    'close': 'last', 'volume': 'sum'
})
monthly.tail(5)

## 第 4 段：`rolling()` 滚动均线

In [ ]:
byd_idx['ma5'] = byd_idx['close'].rolling(window=5).mean()
byd_idx['ma20'] = byd_idx['close'].rolling(window=20).mean()
byd_idx['ma60'] = byd_idx['close'].rolling(window=60).mean()

df_plot = byd_idx.tail(180)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_plot.index, df_plot['close'], label='收盘', color='black', alpha=0.6)
ax.plot(df_plot.index, df_plot['ma5'],  label='MA5',  color=_style.COLORS['ma_short'])
ax.plot(df_plot.index, df_plot['ma20'], label='MA20', color=_style.COLORS['ma_long'])
ax.plot(df_plot.index, df_plot['ma60'], label='MA60', color='purple', alpha=0.6)
ax.legend()
ax.set_title('比亚迪 移动平均线')
plt.show()

**新手陷阱**：rolling 的前 N-1 个值是 NaN（N = 窗口大小），plot 时 matplotlib 自动忽略 NaN。但做策略信号时要注意，这 N-1 天不能用。

## 第 5 段：停牌日 NaN 处理

A 股停牌日（公司公告、重大事件、临停）在 akshare 数据里有时表现为：
- 完全没有这一天的行（最常见）
- 行存在但 volume=0

偶尔我们 resample 后会产生 NaN，要处理。

In [ ]:
# 演示：人为造一段 NaN，练习 fillna / interpolate
demo = byd_idx['close'].tail(30).copy()
demo.iloc[10:13] = np.nan  # 模拟 3 天停牌

fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
demo.plot(ax=axes[0], title='原始（含 NaN）', color='gray')
demo.ffill().plot(ax=axes[1], title='ffill（用前一日填充）', color='blue')
demo.interpolate().plot(ax=axes[2], title='interpolate（线性插值）', color='red')
plt.tight_layout(); plt.show()

**何时用哪个？**
- `fillna(0)`：成交量 0 是真实的（停牌没成交），用这个
- `.ffill()`（forward fill）：价格停牌期间"沿用前一日"，回测常用
- `interpolate()`：图形平滑用，**回测不能用**（会有未来函数嫌疑）
- `dropna()`：直接删掉，适合"不关心这段"的统计分析

## 第 6 段：随堂小练

### 小练：画"收盘价 + 20 日成交量均线"双轴图

In [ ]:
# TODO: 你的代码（约 8 行）
# 1) 算出 byd 的 volume_ma20
# 2) 取最近 180 天
# 3) 左轴画 close（蓝）、右轴画 volume + volume_ma20（橙/绿）
# 提示：ax2 = ax1.twinx() 共享 x 轴

## 第 7 段：课后练习 + 下节预告

### 📝 `exercises/ex03.py`
1. 写 `monthly_stats(df)` 返回每月（open first / high max / low min / close last / volume sum / 涨跌幅 %）的 DataFrame
2. 写 `rolling_sharpe(df, window=20)` 简单版夏普（每日收益 mean / std × sqrt(252)）的滚动序列
3. 对比比亚迪 2024 年 MA20 上穿 MA60（金叉）与下穿（死叉）的次数，标出每次日期

### 🔮 下节 L04：数据清洗 + 复权 + 多股对齐
本节最硬核（75-90 min）。你将理解为什么"不复权" K 线会有假跳空、为什么多股对比必须先对齐日期。

## 第 8 段：Jupyter tip 🔧
- `!pip list | grep pandas`：在 notebook 内跑 shell 命令（! 开头）
- `%load` 一行魔法：把外部文件加载进 cell，例 `%load learning/phase1_foundation/_data.py`
- `ax.xaxis.set_major_formatter()`：格式化日期轴，避免"2022-01-04 00:00:00" 这种丑陋显示